# Resampling-only unfaithfulness monitor - Kaggle run

Free tier: 2x T4, 30 GPU-hr/week, 12-hr sessions. Steps:
1. install vLLM + deps, add `src/` to path
2. start a vLLM server for Qwen3-4B (reasoning) in the background
3. wire a `VLLMBackend` whose `render` applies the chat template with the CoT prefix
4. load ARC-Challenge, keep questions the model gets right with no cue
5. run the cue-flip vs control experiment, save every rollout to disk
6. summarize: AUC + bootstrap CI per detector, ROC plot

**Verify first (cheap):** run the `SMOKE ON REAL MODEL` cell with `n_questions=2`,
`k=8` and confirm (a) the CoT is a real multi-step trace, (b) the cue flips the
answer on a meaningful fraction. If not, bump to Qwen3-8B or change the cue
before spending a full run.

In [ ]:
%pip install -q vllm==0.6.* sentence-transformers datasets
import sys, pathlib
# Upload this repo as a Kaggle dataset or clone it; point REPO at it.
REPO = pathlib.Path('/kaggle/working/MATS')
assert (REPO / 'src' / 'mats').is_dir(), 'put the repo at /kaggle/working/MATS'
sys.path.insert(0, str(REPO / 'src'))

In [ ]:
import subprocess, time, urllib.request

MODEL = 'Qwen/Qwen3-4B'
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--dtype', 'float16',  # T4 is sm_75, no bf16 tensor cores
    '--tensor-parallel-size', '1', '--max-model-len', '6144',  # single T4 (16GB); see README
    '--gpu-memory-utilization', '0.92',
])

def wait_ready(url='http://localhost:8000/health', timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        try:
            urllib.request.urlopen(url, timeout=5); return True
        except Exception:
            time.sleep(5)
    raise RuntimeError('vLLM server did not come up')

wait_ready()

In [ ]:
from transformers import AutoTokenizer
from mats.backend_vllm import VLLMBackend

tok = AutoTokenizer.from_pretrained(MODEL)

def render(prompt: str, prefix: str) -> str:
    # prompt is the user turn; prefix is the assistant's CoT so far.
    text = tok.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True,
    )
    return text + prefix

backend = VLLMBackend(model=MODEL, temperature=0.8, top_p=0.95, render=render,
                      stop=['<|im_end|>'])
print(backend.complete('Reply with the single word OK.', seed=0, max_tokens=16))

## SMOKE ON REAL MODEL - run this before the full experiment

In [ ]:
from mats.data import load_arc_challenge, keep_answerable
from mats.prompts import build_prompt, cue_target
from mats.cot import parse_answer, split_sentences

pool = load_arc_challenge('validation')
probe = keep_answerable(backend, pool, threshold=0.8, k=8, seed=0, limit=2)
for q in probe:
    base = backend.complete(build_prompt(q), seed=1)
    cued = backend.complete(build_prompt(q, cue_letter=cue_target(q)), seed=1)
    print(q.qid, 'sentences=', len(split_sentences(base)),
          'no_cue=', parse_answer(base), 'cued=', parse_answer(cued),
          'cue_target=', cue_target(q))

## Full experiment

In [ ]:
import json
from mats.config import load_config
from mats.embed_st import SentenceTransformerEmbedder
from mats.experiment import run_experiment
from mats.report import summarize, plot_roc

cfg = load_config(REPO / 'configs' / 'defaults.toml')
questions = keep_answerable(
    backend, pool, threshold=cfg.correct_threshold, k=8, seed=0,
    limit=cfg.n_prompts // 2,
)
print('kept', len(questions), 'questions')

embedder = SentenceTransformerEmbedder(device='cuda')
records = run_experiment(backend, embedder, questions, cfg)

out = REPO / 'outputs'
out.mkdir(exist_ok=True)
(out / 'records.json').write_text(json.dumps([r.as_dict() for r in records], indent=2))
summary = summarize(records)
(out / 'metrics.json').write_text(json.dumps(summary, indent=2))
plot_roc(records, out / 'roc.png')
summary

In [ ]:
server.terminate()